In [2]:
import requests, json, time, pathlib

BASE = "https://classes.cornell.edu/api/2.0"

r = requests.get(f"{BASE}/search/classes.json",
                 params={"roster": "FA25", "subject": "CS"})
r.raise_for_status()
print(json.dumps(r.json(), indent=2)[:3000])   # inspect the real structure

{
  "status": "success",
  "data": {
    "classes": [
      {
        "strm": 2909,
        "crseId": 358526,
        "crseOfferNbr": 1,
        "subject": "CS",
        "catalogNbr": "1110",
        "titleShort": "Intro Computing: Design & Dev",
        "titleLong": "Introduction to Computing: A Design and Development Perspective",
        "crseAttrs": [
          {
            "crseAttr": "0012",
            "crseAttrValue": "SMR-AS",
            "descr": "AS-Symbolic & Mathematic Rsng",
            "descrformal": "AS-Symbolic and Mathematical Reasoning",
            "attrDescr": "Distribution Requirements",
            "attrDescrShort": "DistReq"
          },
          {
            "crseAttr": "0012",
            "crseAttrValue": "STA-IL",
            "descr": "IL-Statistics Requirement",
            "descrformal": "IL-Statistics Requirement",
            "attrDescr": "Distribution Requirements",
            "attrDescrShort": "DistReq"
          }
        ],
        "crseAttrValueG

successfull response

In [3]:
resp = r.json()                  # the whole response dict
classes = resp["data"]["classes"]  # list of courses
print(classes[0].keys())         # fields available on one course

dict_keys(['strm', 'crseId', 'crseOfferNbr', 'subject', 'catalogNbr', 'titleShort', 'titleLong', 'crseAttrs', 'crseAttrValueGroups', 'enrollGroups', 'description', 'catalogBreadth', 'catalogDistr', 'catalogEnrollmentPriority', 'catalogLang', 'catalogForbiddenOverlaps', 'catalogAttribute', 'catalogWhenOffered', 'catalogComments', 'catalogPrereqCoreq', 'catalogPrereq', 'catalogCoreq', 'catalogFee', 'catalogSatisfiesReq', 'catalogPermission', 'catalogCourseSubfield', 'catalogOutcomes', 'acadCareer', 'acadGroup', 'catalogName', 'catalogNextOffered', 'lastTermsOffered', 'catalogCredits', 'catalogGradeOption', 'catalogLocation', 'catalogCrosslisted', 'catalogCrosslistings', 'catalogComeets', 'exploreCriteriaIds'])


In [4]:
import pandas as pd

rows = []
for c in classes:
    rows.append({
        "term": c.get("strm"),
        "crseId": c.get("crseId"),
        "subject": c.get("subject"),
        "catalogNbr": c.get("catalogNbr"),
        "title": c.get("titleLong"),
        "description": c.get("description"),
        "outcomes": c.get("catalogOutcomes"),
        "college": c.get("acadGroup"),
        "career": c.get("acadCareer"),
        "credits": c.get("catalogCredits"),
        "crosslisted": c.get("catalogCrosslisted"),
        "crosslistings": c.get("catalogCrosslistings"),
    })
df = pd.DataFrame(rows)

print(df["crosslisted"].value_counts(dropna=False))
print(df["description"].isna().mean())   # share with no description
print(df[["subject", "catalogNbr", "crosslisted", "crosslistings"]].head(10))

crosslisted
None    97
Name: count, dtype: int64
0.0
  subject catalogNbr crosslisted crosslistings
0      CS       1110        None          None
1      CS       1112        None          None
2      CS       1132        None          None
3      CS       1133        None          None
4      CS       1710        None          None
5      CS       1998        None          None
6      CS       2024        None          None
7      CS       2110        None          None
8      CS       2112        None          None
9      CS       2800        None          None


In [5]:
import re

text = (df["title"].fillna("") + " " + df["description"].fillna(""))
kw = re.compile(r"machine learning|artificial intelligence|deep learning|"
                r"neural network|large language model|natural language processing|"
                r"reinforcement learning|computer vision", re.I)
ai_abbrev = re.compile(r"\bAI\b")   # case-sensitive, avoids the word "ai"
df["is_ai"] = text.apply(lambda t: bool(kw.search(t) or ai_abbrev.search(t)))

In [6]:
subjects = ["CS", "INFO", "ORIE", "STSCI", "ECE"]
frames = []
for s in subjects:
    r = requests.get(f"{BASE}/search/classes.json",
                     params={"roster": "FA25", "subject": s})
    for c in r.json()["data"]["classes"]:
        frames.append({"subject": c["subject"], "catalogNbr": c["catalogNbr"],
                       "crseId": c["crseId"], "title": c["titleLong"],
                       "description": c["description"]})
    time.sleep(0.5)

multi = pd.DataFrame(frames)
dups = multi.groupby("description").filter(lambda g: g["subject"].nunique() > 1)
print(len(dups))
print(dups.sort_values("description")[["subject", "catalogNbr", "crseId", "title"]].head(20))

58
    subject catalogNbr  crseId  \
199    ORIE       5260  374759   
256   STSCI       5050  373883   
224    ORIE       6731  374677   
243   STSCI       4050  373882   
94       CS       7800  365623   
26       CS       4754  367112   
159    INFO       6420  367731   
114    INFO       4410  367112   
107    INFO       3900  373443   
241   STSCI       3900  373443   
37       CS       5306  367819   
134    INFO       5306  367819   
186    ORIE       5140  353269   
301     ECE       5120  353269   
158    INFO       6410  366355   
51       CS       5682  366355   
33       CS       4999  358600   
120    INFO       4900  358814   
247   STSCI       4630  352214   
267   STSCI       5630  371816   

                                                 title  
199             Special Topics in Quantitative Finance  
256          Modern Regression Models for Data Science  
224                    LLMs for Better Decision-Making  
243          Modern Regression Models for Data Science

In [7]:
blank = multi["description"].fillna("").str.strip() == ""
print(blank.sum(), len(multi))

5 333


In [8]:
print(len(multi),
      multi["crseId"].nunique(),
      multi.groupby(["title", "description"]).ngroups)

333 308 267


In [9]:
text = multi["title"].fillna("") + " " + multi["description"].fillna("")
multi["is_ai"] = text.apply(lambda t: bool(kw.search(t) or ai_abbrev.search(t)))

versions = {
    "no dedupe": multi,
    "crseId": multi.drop_duplicates("crseId"),
    "title+desc": multi.drop_duplicates(["title", "description"]),
}
for name, d in versions.items():
    print(name, len(d), d["is_ai"].sum(), round(d["is_ai"].mean(), 3))

no dedupe 333 70 0.21
crseId 308 62 0.201
title+desc 267 53 0.199


In [10]:
b = versions["crseId"]
extra = b[b.duplicated(["title", "description"], keep=False)]
print(extra.sort_values("title")[["subject", "catalogNbr", "crseId", "title"]])

    subject catalogNbr  crseId  \
129    INFO       5170  375035   
97     INFO       1170  375034   
61       CS       5820  371771   
79       CS       6820  358642   
202    ORIE       5340  371850   
..      ...        ...     ...   
131    INFO       5235  373213   
146    INFO       5431  372220   
116    INFO       4430  367429   
187    ORIE       5160  373046   
174    ORIE       4160  372982   

                                                 title  
129                                AI in Organizations  
97                                 AI in Organizations  
61                              Analysis of Algorithms  
79                              Analysis of Algorithms  
202  Applications of Optimization: Modeling and Com...  
..                                                 ...  
131      Sustainable Urban and Energy Delivery Systems  
146                               Teams and Technology  
116                               Teams and Technology  
187                  